# 評定型（トラディショナル）コンジョイント分析

**評定型コンジョイント分析（rating-based / traditional conjoint analysis）** は、回答者に個々のプロファイル（属性の組み合わせ）を1つずつ提示し、「どの程度好ましいか」を評点（例：1〜7点のリッカート尺度、0〜100点の購入意向）で回答させる、最も古典的なコンジョイント分析の方式である。

得られた評点を目的変数、各属性の水準をダミー変数化した説明変数として **線形回帰（OLS）** を行うことで、部分効用を推定する。

## モデル

回答者がプロファイル$i$に与える評点$y_i$を以下のように表すとする。

$$
y_i = \beta_0 + \sum_{k=1}^{K} \sum_{l=1}^{L_k - 1} \beta_{kl}\, d_{kl}(i) + \varepsilon_i,
\quad \varepsilon_i \sim N(0, \sigma^2)
$$

ここで

- $d_{kl}(i)$：属性$k$の水準$l$に対応する符号化変数
- $\beta_{kl}$：その水準の部分効用
- $\beta_0$：切片（全体平均効用）


## ダミーコーディングと効果コーディング

属性はカテゴリ変数なので、回帰に使う前に数値化する必要がある。代表的な方式が2つある。

- **ダミーコーディング（dummy coding）**：ある水準を基準（リファレンス）として$0/1$で表す。基準水準の部分効用は$0$に固定され、他の水準の係数は基準水準からの差分として解釈される。
- **効果コーディング（effects coding）**：基準水準を$-1$、対象水準を$+1$、その他を$0$で表す。

この符号化では

$$
\sum_{l=1}^{L_k} \beta_{kl} = 0 \quad (\text{各属性 } k \text{ について})
$$

という制約のもとで各水準の部分効用が推定されるため、**切片$\beta_0$がプロファイル全体の平均効用**として解釈できる。

コンジョイント分析では、部分効用を「属性内水準の平均からの乖離」として解釈したいことが多く、水準間の比較が基準水準に依存しない効果コーディングが標準的に用いられる。

## 属性重要度

各属性$k$の部分効用の値域（レンジ）

$$
\text{range}_k = \max_l \beta_{kl} - \min_l \beta_{kl}
$$

を全属性で正規化したものを、属性の相対的な重要度とする。

$$
\text{Importance}_k = \frac{\text{range}_k}{\sum_{j=1}^{K} \text{range}_j}
$$

## 実装例

4属性からなる架空の飲料製品について、真の部分効用を設定してプロファイルへの評点データを生成し、効果コーディング＋OLSで部分効用を復元できるか確認する。

In [1]:
import itertools
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)

attributes = {
    "価格":     ["1,000円", "1,500円", "2,000円"],
    "容量":     ["500ml", "1000ml"],
    "ブランド":  ["A社", "B社", "C社"],
    "パッケージ": ["缶", "瓶", "ペットボトル"],
}

# 真の部分効用（各属性内の和が0になるように設定）
true_partworths = {
    "価格":     {"1,000円": 1.5, "1,500円": 0.0, "2,000円": -1.5},
    "容量":     {"500ml": -0.5, "1000ml": 0.5},
    "ブランド":  {"A社": 0.8, "B社": 0.2, "C社": -1.0},
    "パッケージ": {"缶": -0.3, "瓶": 0.6, "ペットボトル": -0.3},
}
intercept_true = 5.0

profiles = list(itertools.product(*attributes.values()))
df = pd.DataFrame(profiles, columns=attributes.keys())

def true_utility(row):
    return intercept_true + sum(
        true_partworths[attr][row[attr]] for attr in attributes
    )

df["true_utility"] = df.apply(true_utility, axis=1)
df["rating"] = df["true_utility"] + rng.normal(0, 0.3, size=len(df))
df.head()


,価格,容量,ブランド,パッケージ,true_utility,rating
0,"1,000円",500ml,A社,缶,6.5,6.537719
1,"1,000円",500ml,A社,瓶,7.4,7.360369
2,"1,000円",500ml,A社,ペットボトル,6.5,6.692127
3,"1,000円",500ml,B社,缶,5.9,5.931470
4,"1,000円",500ml,B社,瓶,6.8,6.639299


In [ ]:
# 効果コーディング（各属性の最終水準を基準とし、-1/0/+1で符号化）
def effects_code(df, attributes):
    X = pd.DataFrame(index=df.index)
    for attr, levels in attributes.items():
        base = levels[-1]  # 最終水準を基準に
        for level in levels[:-1]:
            col = f"{attr}::{level}"
            X[col] = 0
            X.loc[df[attr] == level, col] = 1
            X.loc[df[attr] == base, col] = -1
    return X

X = effects_code(df, attributes)
X.insert(0, "intercept", 1)
X.head()

,intercept,"価格::1,000円","価格::1,500円",容量::500ml,ブランド::A社,ブランド::B社,パッケージ::缶,パッケージ::瓶
0,1,1,0,1,1,0,1,0
1,1,1,0,1,1,0,0,1
2,1,1,0,1,1,0,-1,-1
3,1,1,0,1,0,1,1,0
4,1,1,0,1,0,1,0,1


In [3]:
import statsmodels.api as sm

model = sm.OLS(df["rating"], X)
result = model.fit()
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 rating   R-squared:                       0.974
Model:                            OLS   Adj. R-squared:                  0.970
Method:                 Least Squares   F-statistic:                     249.9
Date:                Sat, 29 Aug 2026   Prob (F-statistic):           2.16e-34
Time:                        16:57:37   Log-Likelihood:                0.78245
No. Observations:                  54   AIC:                             14.44
Df Residuals:                      46   BIC:                             30.35
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      5.0347      0.035    143.181      0.000       4.964       5.106
価格::1,000円     1.3800      0.050     27.750      0.000       1.280       1.480
価格::1,500円    -0.0146      0.050     -0.294      0.770      -0.115       0.085
容量::500ml     -0.4360      0.035    -12.399      0.000      -0.507      -0.365
ブランド::A社       0.7996      0.050     16.079      0.000       0.699       0.900
ブランド::B社       0.1826      0.050      3.671      0.001       0.082       0.283
パッケージ::缶      -0.3186      0.050     -6.407      0.000      -0.419      -0.218
パッケージ::瓶       0.5832      0.050     11.727      0.000       0.483       0.683
==============================================================================
Omnibus:                        0.560   Durbin-Watson:                   1.787
Prob(Omnibus):                  0.756   Jarque-Bera (JB):                0.587
Skew:                          -0.227   Prob(JB):                        0.746
Kurtosis:                       2.766   Cond. No.                         1.73
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [4]:
# 基準水準の部分効用は、他の水準の推定値の符号反転和として復元する
coef = result.params.drop("intercept")

partworths_hat = {"intercept": result.params["intercept"]}
for attr, levels in attributes.items():
    attr_coefs = {
        level: coef[f"{attr}::{level}"]
        for level in levels[:-1]
    }
    base_level = levels[-1]
    attr_coefs[base_level] = -sum(attr_coefs.values())
    partworths_hat[attr] = attr_coefs

pd.DataFrame(partworths_hat)

,intercept,価格,容量,ブランド,パッケージ
"1,000円",5.034727,1.379978,NaN,NaN,NaN
"1,500円",5.034727,-0.014628,NaN,NaN,NaN
"2,000円",5.034727,-1.365350,NaN,NaN,NaN
500ml,5.034727,NaN,-0.436004,NaN,NaN
1000ml,5.034727,NaN,0.436004,NaN,NaN
A社,5.034727,NaN,NaN,0.799571,NaN
B社,5.034727,NaN,NaN,0.182577,NaN
C社,5.034727,NaN,NaN,-0.982149,NaN
缶,5.034727,NaN,NaN,NaN,-0.318596
瓶,5.034727,NaN,NaN,NaN,0.583162


推定された部分効用$\hat\beta$は、ノイズの範囲内で真の部分効用（価格：1.5 / 0.0 / -1.5、容量：-0.5 / 0.5、ブランド：0.8 / 0.2 / -1.0、パッケージ：-0.3 / 0.6 / -0.3、切片：5.0）を概ね再現できている。

In [5]:
# 属性重要度
importance = {}
for attr, levels in attributes.items():
    values = list(partworths_hat[attr].values())
    importance[attr] = max(values) - min(values)

total = sum(importance.values())
importance_pct = {k: v / total for k, v in importance.items()}
pd.Series(importance_pct).sort_values(ascending=False).rename("importance")


価格       0.435710
ブランド     0.282776
パッケージ    0.143118
容量       0.138396
Name: importance, dtype: float64

## 評定型コンジョイント分析の限界

- 個々のプロファイルを独立に評価させるため、**属性間のトレードオフ（選好の一貫性）が反映されにくい**（実際の購買では選択肢の中から1つを選ぶ相対比較が起きる）
- 評点尺度の使い方に回答者間で個人差がある（scale usage heterogeneity）
- 属性数・水準数が多いと、提示するプロファイル数が増え回答負荷が高くなる

これらの限界から、実務では **選択型コンジョイント分析（CBC）** が主流になっている。